In [62]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.formula.api as sm
import sklearn.preprocessing as preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import os

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [63]:
folder = '/content/drive/MyDrive/data_science'
df = pd.read_csv('/content/drive/MyDrive/data_science/output/dataset_final.csv')


In [64]:
datos = df.copy()

# **Variable objetivo**
La variable obejtivo es "target_ampliacion_monto" basandose en el objetivo del trabajo.

Se utilizaran modelos de categorizacion para poder identificar contratos que podrían requerir un seguimiento más cercano durante su ejecución apartir de su adjudicacion.

Se eliminan algunas variables que no son relevantes para el modelo o que no son soportados con estandarizacion o algunos algoritmos.

In [65]:
columnas_drop = [
    'award_id',
    'release_id',
    'contract_id',
    'ruc',
    'proveedor',
    'fecha_fin_lic',
    'fecha_ini_lic',
    'fecha_firma',
    'fecha_pub_lic',
    'n_enmiendas',
    'hubo_ampl_plazo',
    'monto_ampliado_total',
    'dias_ampliacion_total',
    'dias_hasta_enmienda',
    'ratio_ampliacion'
]

datos = datos.drop(
    columns=columnas_drop,
    errors='ignore'
)

In [66]:

datos['fecha_adj'] = pd.to_datetime(
    datos['fecha_adj'],
    errors='coerce'
)

datos = datos.sort_values('fecha_adj').reset_index(drop=True)

In [67]:
objetivo = 'target_ampliacion_monto'

# **Revision de datos**
tratamiento de valores faltantes o irregulares

In [68]:
for col in datos.columns:
  print(f"{col}")
  print("Cantidad de nulos:", datos[col].isna().sum())
  print("Cantidad de ceros:" + str((datos[col] == 0).sum()) + "\n")

fecha_adj
Cantidad de nulos: 0
Cantidad de ceros:0

monto_adj
Cantidad de nulos: 0
Cantidad de ceros:1

log_monto
Cantidad de nulos: 0
Cantidad de ceros:1

anio
Cantidad de nulos: 0
Cantidad de ceros:0

mes
Cantidad de nulos: 0
Cantidad de ceros:0

plazo_dias
Cantidad de nulos: 4
Cantidad de ceros:40

prov_n_previas
Cantidad de nulos: 0
Cantidad de ceros:3594

prov_monto_previo_acum
Cantidad de nulos: 0
Cantidad de ceros:3594

prov_antiguedad_dias
Cantidad de nulos: 0
Cantidad de ceros:3715

n_lotes
Cantidad de nulos: 0
Cantidad de ceros:813

monto_estimado_lotes
Cantidad de nulos: 0
Cantidad de ceros:813

metodo_contratacion
Cantidad de nulos: 0
Cantidad de ceros:0

det_metodo_contratacion
Cantidad de nulos: 0
Cantidad de ceros:0

categoria_principal
Cantidad de nulos: 0
Cantidad de ceros:0

monto_licitacion
Cantidad de nulos: 0
Cantidad de ceros:3

monto_presupuesto
Cantidad de nulos: 813
Cantidad de ceros:0

ratio_adj_vs_licitacion
Cantidad de nulos: 0
Cantidad de ceros:1

brecha_li

In [69]:
median = [
    'plazo_dias',
    'dur_licitacion_dias',
    'dur_consultas_dias',
    'dias_lic_a_adj',
    'monto_presupuesto'
]

In [70]:
for col in median:
    datos[col].fillna(datos[col].median(), inplace=True)
datos['brecha_licitacion_vs_presupuesto'] = datos['monto_presupuesto']/datos['monto_licitacion']
datos[datos['dur_licitacion_dias'] < 0 ]['dur_licitacion_dias'] = datos['dur_licitacion_dias'].median()
datos[datos['dur_consultas_dias'] < 0 ]['dur_consultas_dias'] = datos['dur_consultas_dias'].median()
datos[datos['velocidad_proceso_dias'] < 0 ]['velocidad_proceso_dias'] = datos['velocidad_proceso_dias'].median()
datos['brecha_licitacion_vs_presupuesto'].replace([np.inf, -np.inf], np.nan, inplace=True)
datos['brecha_licitacion_vs_presupuesto'].fillna(0, inplace=True)
datos['ratio_adj_vs_licitacion'].replace([np.inf, -np.inf], np.nan, inplace=True)
datos['ratio_adj_vs_licitacion'].fillna(0, inplace=True)

/tmp/ipykernel_5496/672834934.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  datos[col].fillna(datos[col].median(), inplace=True)
/tmp/ipykernel_5496/672834934.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  datos[datos['dur_licitacion_dias'] < 0 ]['dur_licitacion_dias'] = datos

In [71]:
for col in datos.columns:
  print(f"{col}")
  print("Cantidad de nulos:", datos[col].isna().sum())
  print("Cantidad de ceros:" + str((datos[col] == 0).sum()) + "\n")

fecha_adj
Cantidad de nulos: 0
Cantidad de ceros:0

monto_adj
Cantidad de nulos: 0
Cantidad de ceros:1

log_monto
Cantidad de nulos: 0
Cantidad de ceros:1

anio
Cantidad de nulos: 0
Cantidad de ceros:0

mes
Cantidad de nulos: 0
Cantidad de ceros:0

plazo_dias
Cantidad de nulos: 0
Cantidad de ceros:40

prov_n_previas
Cantidad de nulos: 0
Cantidad de ceros:3594

prov_monto_previo_acum
Cantidad de nulos: 0
Cantidad de ceros:3594

prov_antiguedad_dias
Cantidad de nulos: 0
Cantidad de ceros:3715

n_lotes
Cantidad de nulos: 0
Cantidad de ceros:813

monto_estimado_lotes
Cantidad de nulos: 0
Cantidad de ceros:813

metodo_contratacion
Cantidad de nulos: 0
Cantidad de ceros:0

det_metodo_contratacion
Cantidad de nulos: 0
Cantidad de ceros:0

categoria_principal
Cantidad de nulos: 0
Cantidad de ceros:0

monto_licitacion
Cantidad de nulos: 0
Cantidad de ceros:3

monto_presupuesto
Cantidad de nulos: 0
Cantidad de ceros:0

ratio_adj_vs_licitacion
Cantidad de nulos: 0
Cantidad de ceros:4

brecha_lici

In [72]:
columnas_no_predictoras = [
    objetivo,
    'fecha_adj'
]

X = datos.drop(columns=columnas_no_predictoras, errors='ignore')

y = datos[objetivo]

print("\nDimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)

print("\nVariables predictoras:")
print(X.columns.tolist())
X.shape


Dimensiones de X: (14268, 24)
Dimensiones de y: (14268,)

Variables predictoras:
['monto_adj', 'log_monto', 'anio', 'mes', 'plazo_dias', 'prov_n_previas', 'prov_monto_previo_acum', 'prov_antiguedad_dias', 'n_lotes', 'monto_estimado_lotes', 'metodo_contratacion', 'det_metodo_contratacion', 'categoria_principal', 'monto_licitacion', 'monto_presupuesto', 'ratio_adj_vs_licitacion', 'brecha_licitacion_vs_presupuesto', 'criterio_adj', 'det_criterio_adj', 'dur_licitacion_dias', 'dur_consultas_dias', 'velocidad_proceso_dias', 'dias_pub_a_adj', 'dias_lic_a_adj']


(14268, 24)

In [73]:
categoricas = [
    'metodo_contratacion', 'det_metodo_contratacion', 'categoria_principal', 'criterio_adj', 'det_criterio_adj'
]
X[categoricas].describe()
for col in categoricas:
    print(f"\n--- {col} ---")
    print(X[col].value_counts(dropna=False))


--- metodo_contratacion ---
metodo_contratacion
open      14001
direct      267
Name: count, dtype: int64

--- det_metodo_contratacion ---
det_metodo_contratacion
Contratacion Directa                                            6651
Licitacion Publica Nacional                                     2913
Concurso de Ofertas                                             2764
Locacion de Inmuebles                                            846
Contratacion por Excepcion                                       813
Licitacion Publica Internacional                                  56
BID - Contratacion Directa (CD/SD)                                55
BID - Contratacion por Excepcion                                  29
BID - Concurso de Ofertas (CO/CP)                                 28
BID - Licitacion Publica Nacional                                 16
Seleccion sobre la base la comparacion de las calificaciones      15
BM - Concurso de Ofertas - CO                                     13
BID - Li

Aplicar One-Hot Encoding a las variables categóricas, si bien det_metodo_contratacion tiene muchas variables unicas, todavia se considera aceptable la aplicacion del metodo de encoding al tener una gran cantidad de filas para el entrenamiento.

In [74]:
X_encoded = pd.get_dummies(X, columns=categoricas, drop_first=True)
print("Dimensiones de desarrollo codificado:", X_encoded.shape)
X_encoded.info()


Dimensiones de desarrollo codificado: (14268, 51)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14268 entries, 0 to 14267
Data columns (total 51 columns):
 #   Column                                                                                Non-Null Count  Dtype  
---  ------                                                                                --------------  -----  
 0   monto_adj                                                                             14268 non-null  int64  
 1   log_monto                                                                             14268 non-null  float64
 2   anio                                                                                  14268 non-null  int64  
 3   mes                                                                                   14268 non-null  int64  
 4   plazo_dias                                                                            14268 non-null  float64
 5   prov_n_previas                 

# **Division**
Se divide las filas cronologicamente de acuerdo a fecha_adj, 80% para el desarrollo y 20% para la prueba

In [75]:
fechas = datos['fecha_adj']
corte = int(len(datos) * 0.80)

X_desarrollo = X_encoded.iloc[:corte].copy()
y_desarrollo = y.iloc[:corte].copy()

X_prueba = X_encoded.iloc[corte:].copy()
y_prueba = y.iloc[corte:].copy()

# 9. Mostrar los resultados
print("\nDimensiones:")
print("X desarrollo:", X_desarrollo.shape)
print("y desarrollo:", y_desarrollo.shape)
print("X prueba:", X_prueba.shape)
print("y prueba:", y_prueba.shape)

print("\nPeríodo de desarrollo:")
print(fechas.iloc[0], "a", fechas.iloc[corte - 1])

print("\nPeríodo de prueba:")
print(fechas.iloc[corte], "a", fechas.iloc[-1])

print("\nDistribución del objetivo en desarrollo:")
print(y_desarrollo.value_counts(normalize=True))

print("\nDistribución del objetivo en prueba:")
print(y_prueba.value_counts(normalize=True))


Dimensiones:
X desarrollo: (11414, 51)
y desarrollo: (11414,)
X prueba: (2854, 51)
y prueba: (2854,)

Período de desarrollo:
2010-06-14 21:03:59 a 2011-12-06 18:27:04

Período de prueba:
2011-12-06 18:35:06 a 2013-03-05 16:00:00

Distribución del objetivo en desarrollo:
target_ampliacion_monto
0.0    0.886806
1.0    0.113194
Name: proportion, dtype: float64

Distribución del objetivo en prueba:
target_ampliacion_monto
0.0    0.915557
1.0    0.084443
Name: proportion, dtype: float64


# **Escalado de variables**

In [76]:
# Identificar y seleccionar solo columnas numéricas y booleanas para el escalado
columnas_numericas = X_desarrollo.select_dtypes(include=[np.number, bool]).columns

scaler = StandardScaler()
X_desarrollo_scaled = scaler.fit_transform(X_desarrollo)
X_prueba_scaled = scaler.transform(X_prueba)

print("Dimensiones de desarrollo escalado:", X_desarrollo_scaled.shape)
print("Dimensiones de prueba escalado:", X_prueba_scaled.shape)

Dimensiones de desarrollo escalado: (11414, 51)
Dimensiones de prueba escalado: (2854, 51)
